In [ ]:
# 1. Installer og importer nødvendige pakker
!pip install --upgrade openai --quiet
import pandas as pd
import re
from openai import OpenAI

# 2. Sett API-nøkkel og lag klient
client = OpenAI(api_key="APIKEY_HER")  # ← sett din faktiske nøkkel her

# 3. Les CSV-data
df = pd.read_csv("ASL_LIFEPRINT_CLEAN.csv")
df = df.dropna()
df = df.reset_index(drop=True)

# 4. Resultatbuffer
results = []

# 5. Funksjon for å prosessere én batch (10 linjer)
def process_batch(batch):
    prompt = (
        "Du er en ekspert på American Sign Language (ASL). Her er 10 tekstlinjer fra en database med ustrukturert ASL-innhold.\n"
        "For hver linje skal du returnere KUN den rensede og standardiserte ASL-glossen (store bokstaver, ASL-rekkefølge), "
        "eller tom streng hvis linjen ikke inneholder en gyldig ASL-struktur.\n\n"
    )
    numbered = "\n".join([f"{i+1}. {text}" for i, text in enumerate(batch)])
    full_prompt = prompt + numbered

    response = client.chat.completions.create(
        model="gpt-4-turbo",
        messages=[
            {"role": "system", "content": "Du er en hjelpsom og presis lingvist med ekspertise i ASL."},
            {"role": "user", "content": full_prompt}
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content
    cleaned = []
    for i in range(1, 11):
        match = re.search(rf"{i}\.\s*(.*)", answer)
        cleaned.append(match.group(1).strip() if match else "")
    return cleaned

# 6. Gå gjennom datasettet i batcher
for i in range(0, len(df), 10):
    batch = df.iloc[i:i+10]['ASL'].tolist()
    print(f"🔄 Sender rader {i+1}–{i+len(batch)} til GPT...")
    cleaned = process_batch(batch)
    for original, cleaned_text in zip(batch, cleaned):
        results.append({"original": original, "asl_clean": cleaned_text})

# 7. Lagre resultat
result_df = pd.DataFrame(results)
result_df.to_csv("asl_gpt_cleaned.csv", index=False)
print("✅ Ferdig! Fil lagret som asl_gpt_cleaned.csv")



🔄 Sender rader 1–10 til GPT...
🔄 Sender rader 11–20 til GPT...
🔄 Sender rader 21–30 til GPT...
🔄 Sender rader 31–40 til GPT...
🔄 Sender rader 41–50 til GPT...
🔄 Sender rader 51–60 til GPT...
🔄 Sender rader 61–70 til GPT...
🔄 Sender rader 71–80 til GPT...
🔄 Sender rader 81–90 til GPT...
🔄 Sender rader 91–100 til GPT...
🔄 Sender rader 101–110 til GPT...
🔄 Sender rader 111–120 til GPT...
🔄 Sender rader 121–130 til GPT...
🔄 Sender rader 131–140 til GPT...
🔄 Sender rader 141–150 til GPT...
🔄 Sender rader 151–160 til GPT...
🔄 Sender rader 161–170 til GPT...
🔄 Sender rader 171–180 til GPT...
🔄 Sender rader 181–190 til GPT...
🔄 Sender rader 191–200 til GPT...
🔄 Sender rader 201–210 til GPT...
🔄 Sender rader 211–220 til GPT...
🔄 Sender rader 221–230 til GPT...
🔄 Sender rader 231–240 til GPT...
🔄 Sender rader 241–250 til GPT...
🔄 Sender rader 251–260 til GPT...
🔄 Sender rader 261–270 til GPT...
🔄 Sender rader 271–280 til GPT...
🔄 Sender rader 281–290 til GPT...
🔄 Sender rader 291–300 til GPT...


In [4]:
import pandas as pd

# Les inn datasettet
df = pd.read_csv("asl_gpt_cleaned.csv")

# Rens NaN i 'asl_clean' kolonnen
df['asl_clean'] = df['asl_clean'].fillna("")

# Tell antall ord per rad
df['gloss_count'] = df['asl_clean'].str.split().apply(len)

# Del i to grupper
df_7plus = df[df['gloss_count'] >= 7].drop(columns=['gloss_count'])
df_under7 = df[df['gloss_count'] < 7].drop(columns=['gloss_count'])

# Lagre begge til filer
df_7plus.to_csv("asl_7plus.csv", index=False)
df_under7.to_csv("asl_under7.csv", index=False)

# Bekreft
print(f"✅ Lagret {len(df_7plus)} rader til 'asl_7plus.csv'")
print(f"✅ Lagret {len(df_under7)} rader til 'asl_under7.csv'")


✅ Lagret 122 rader til 'asl_7plus.csv'
✅ Lagret 1632 rader til 'asl_under7.csv'


In [5]:
import pandas as pd

# Les filen
df = pd.read_csv("asl_gpt_cleaned.csv")

# Sørg for at kolonnen er tekst, og fyll inn tomme med ""
df['asl_clean'] = df['asl_clean'].fillna("").astype(str)

# Tell antall tegn (characters) i hver rad
df['char_count'] = df['asl_clean'].apply(len)

# Del datasettet
df_7plus = df[df['char_count'] >= 7].drop(columns=['char_count'])
df_under7 = df[df['char_count'] < 7].drop(columns=['char_count'])

# Lagre som to nye filer
df_7plus.to_csv("asl_7plus_chars.csv", index=False)
df_under7.to_csv("asl_under7_chars.csv", index=False)

# Print bekreftelse
print(f"✅ {len(df_7plus)} rader med 7+ tegn lagret i 'asl_7plus_chars.csv'")
print(f"✅ {len(df_under7)} rader med <7 tegn lagret i 'asl_under7_chars.csv'")


✅ 1414 rader med 7+ tegn lagret i 'asl_7plus_chars.csv'
✅ 340 rader med <7 tegn lagret i 'asl_under7_chars.csv'
